# 01 — Setup y extracción del informe FEMSA 2024

Prepara todo lo demás: monta Drive, clona el repo, instala dependencias con las versiones fijadas del proyecto, descarga el informe anual FEMSA 2024 (edición en inglés, 151 páginas) y extrae el texto por página + las tripletas por reglas. **No necesita Ollama** (la extracción es determinista, sin LLM).

La extracción es **autocontenida**: la lógica de descarga y extracción (la de `pipeline/extract.py`, citado como procedencia) va copiada inline en las celdas; el notebook no ejecuta ningún script del repo.

**Orden:** corre este notebook después del 00 (o sin él: este notebook no usa Ollama, pero verifica que el servidor esté arriba para los que siguen).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Dependencias (versiones fijadas del proyecto)

Se instalan las mismas versiones que usaron las corridas originales (ver `repo/pyproject.toml` y los READMEs de `memo/`, `rag_baseline/` y `agentic_rag/`). En Colab, `torch` se instala sin el índice CPU: pip resuelve la rueda CUDA compatible con la T4. Si Colab ya trae un torch compatible, pip lo deja como está.

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 sentence-transformers==6.3.0 numpy>=2.5.5 pypdf>=7.1.1 neo4j>=6.4.1 rank-bm25==0.2.2 langchain==1.5.2 langgraph==1.2.24 langchain-core==1.8.5 langchain-huggingface==1.3.0 langchain-ollama

## Verificar Ollama (opcional aquí, obligatorio después)

Este notebook no usa Ollama, pero los notebooks 04–08 sí. Si el servidor no está arriba, corre el notebook `00_ollama_setup.ipynb` (o deja que cada notebook levante el suyo con su bloque idempotente).

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
        print("Ollama responde en localhost:11434 (status", r.status, ")")
except Exception as e:
    print("Ollama NO responde:", e)
    print("Corre 00_ollama_setup.ipynb, o sigue: los notebooks 04-08 levantan el servidor solos.")

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive.
3. **Solo si no lo subes**, generarlo aquí mismo con la lógica de extracción inline (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal. La lógica es la de `pipeline/extract.py` del repo, copiada en la celda: el notebook no ejecuta scripts del repo.

`triples_seed.json`, `eval_questions.json` y `eval_questions_100.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive.")
    print("Si no lo subes, el notebook lo genera aquí mismo con la lógica inline (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar los chunks aquí mismo...")
        print("(lógica de extracción inline: descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        # El PDF está cifrado con AES (contraseña vacía): pypdf necesita el
        # paquete opcional `cryptography` para leerlo, que `pip install pypdf`
        # no trae. Se verifica aquí para que el último recurso funcione.
        try:
            import pypdf
            import cryptography
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1", "cryptography>=3.1"], check=True)
            import pypdf
            import cryptography
        import urllib.request
        from pypdf import PdfReader

        # Lógica de extracción del informe, copiada inline: este notebook no
        # ejecuta scripts del repo. Solo la edición en inglés: el sitio en
        # español bloquea las descargas directas (WAF), así que la edición en
        # inglés es la que se descarga (mismo informe, mismas cifras).
        _PDF_URL_EN = "https://annualreport.femsa.com/pdf/FEMSA_2024_Annual_Report.pdf"
        _PDF_PATH_EN = Path(DATA_DIR) / "FEMSA_2024_Annual_Report_EN.pdf"
        _SOURCE_LABEL_EN = "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)"

        def _inline_is_valid_pdf(path):
            try:
                return len(PdfReader(str(path)).pages) > 0
            except Exception:
                return False

        def _inline_download_pdf():
            if _PDF_PATH_EN.exists() and _inline_is_valid_pdf(_PDF_PATH_EN):
                print(f"PDF en caché: {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
                return _PDF_PATH_EN
            if _PDF_PATH_EN.exists():
                print(f"Caché inválido, se vuelve a descargar: {_PDF_PATH_EN}")
                _PDF_PATH_EN.unlink()
            Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Descargando {_PDF_URL_EN} ...")
            req = urllib.request.Request(
                _PDF_URL_EN, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(_PDF_PATH_EN, "wb") as fh:
                    fh.write(resp.read())
            if not _inline_is_valid_pdf(_PDF_PATH_EN):
                _PDF_PATH_EN.unlink()
                raise RuntimeError("El archivo descargado no es un PDF válido.")
            print(f"Guardado en {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
            return _PDF_PATH_EN

        def _inline_extract_pages(pdf_path):
            reader = PdfReader(str(pdf_path))
            chunks = []
            for i, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                chunks.append({"page": i, "text": text, "chars": len(text),
                               "fuente": _SOURCE_LABEL_EN})
            return chunks

        try:
            _pdf = _inline_download_pdf()
            _chunks = _inline_extract_pages(_pdf)
            _non_empty = sum(1 for c in _chunks if c["chars"] > 0)
            print(f"Páginas: {len(_chunks)} ({_non_empty} con texto)")
            Path(CHUNKS).parent.mkdir(parents=True, exist_ok=True)
            with open(CHUNKS, "w", encoding="utf-8") as fh:
                for c in _chunks:
                    fh.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"{len(_chunks)} páginas -> {CHUNKS}")
        except Exception as e:
            print("Generación inline falló:", e)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

## Tripletas candidatas por reglas (determinista)

Las 13 reglas deterministas de `pipeline/extract.py` (citado como procedencia), copiadas inline: cada regla solo emite una tripleta si su patrón aparece literalmente en el texto de la página, con la página como fuente. Son **candidatas**: hay que revisarlas antes de cargarlas al grafo (ver `pipeline/README.md`). Si `extracted_triples.jsonl` ya existe en Drive, no se regenera.

In [ ]:
import json, os, re

_TRIPLES_OUT = REPO + "/pipeline/extracted_triples.jsonl"

if os.path.isfile(_TRIPLES_OUT):
    print("extracted_triples.jsonl ya existe, no se regenera:", _TRIPLES_OUT)
else:
    _tc_chunks = [json.loads(l) for l in
                  open(REPO + "/pipeline/data/chunks.jsonl")]
    _tc_label = (_tc_chunks[0]["fuente"] if _tc_chunks else
                 "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)")

    # (patrón, needs opcional, campos de la tripleta). Reglas deterministas:
    # solo emiten si el patrón aparece literalmente en el texto de la página.
    _INLINE_RULES = [
        {"pattern": r"781[,\s]?585",
         "subject": "FEMSA", "subject_type": "Empresa", "relation": "reporta",
         "object": "Ingresos Totales", "object_type": "IndicadorFinanciero",
         "valor": 781585, "unidad": "Ps. millones", "variacion_pct": 11.2,
         "periodo": "2024"},
        {"pattern": r"307[,\s]?197", "needs": r"Proximity\s+Americas",
         "subject": "Proximidad Américas", "subject_type": "Division",
         "relation": "reporta", "object": "Ingresos Totales",
         "object_type": "IndicadorFinanciero",
         "valor": 307197, "unidad": "Ps. millones", "variacion_pct": 10.3,
         "periodo": "2024",
         "nota": "La ontología v1 redondea a Ps. 307,200 M; valor exacto del MD&A."},
        {"pattern": r"23[,\s]?206",
         "subject": "Proximidad Américas", "subject_type": "Division",
         "relation": "mide", "object": "Tiendas",
         "object_type": "MetricaOperativa",
         "valor": 23206, "unidad": "tiendas", "periodo": "2024",
         "nota": "23,206 = tiendas OXXO en México (key data); el total de "
                 "Proximidad Américas es 26,322 puntos de venta."},
        {"pattern": r"1[,\s]?596",
         "subject": "Proximidad Américas", "subject_type": "Division",
         "relation": "mide", "object": "Aperturas Netas",
         "object_type": "MetricaOperativa",
         "valor": 1596, "unidad": "tiendas netas nuevas", "periodo": "2024",
         "nota": "La ontología v1 registra 1,236 aperturas (cifra del "
                 "comunicado 4T24); el informe reporta 1,596 aperturas "
                 "netas. No son la misma métrica: revisar antes de cargar."},
        {"pattern": r"279[,\s]?793", "needs": r"Coca-Cola\s+FEMSA",
         "subject": "Coca-Cola FEMSA", "subject_type": "Division",
         "relation": "reporta", "object": "Ingresos Totales",
         "object_type": "IndicadorFinanciero",
         "valor": 279793, "unidad": "Ps. millones", "variacion_pct": 14.2,
         "periodo": "2024",
         "nota": "La ontología v1 redondea a Ps. 279,800 M; valor exacto del MD&A."},
        {"pattern": r"49[,\s]?755", "needs": r"Proximity\s+Europe",
         "subject": "Proximidad Europa", "subject_type": "Division",
         "relation": "reporta", "object": "Ingresos Totales",
         "object_type": "IndicadorFinanciero",
         "valor": 49755, "unidad": "Ps. millones", "variacion_pct": 14.2,
         "periodo": "2024",
         "nota": "La ontología v1 redondea a Ps. 49,800 M; valor exacto del MD&A."},
        {"pattern": r"79[,\s]?755", "needs": r"Health\s+Division",
         "subject": "Salud", "subject_type": "Division",
         "relation": "reporta", "object": "Ingresos Totales",
         "object_type": "IndicadorFinanciero",
         "valor": 79755, "unidad": "Ps. millones", "variacion_pct": 5.8,
         "periodo": "2024",
         "nota": "La ontología v1 redondea a Ps. 79,700 M; valor exacto del MD&A."},
        {"pattern": r"65[,\s]?365", "needs": r"Fuel",
         "subject": "Combustibles", "subject_type": "Division",
         "relation": "reporta", "object": "Ingresos Totales",
         "object_type": "IndicadorFinanciero",
         "valor": 65365, "unidad": "Ps. millones", "variacion_pct": 11.7,
         "periodo": "2024",
         "nota": "La ontología v1 redondea a Ps. 65,400 M; valor exacto del MD&A."},
        {"pattern": r"8\.6\s*million",
         "subject": "Spin by OXXO", "subject_type": "Formato",
         "relation": "mide", "object": "Usuarios Activos",
         "object_type": "MetricaOperativa",
         "valor": 8.6, "unidad": "millones de usuarios", "variacion_pct": 24.9,
         "periodo": "2024",
         "nota": "El +24.9% viene del comunicado 4T24 (vs 4T23); no "
                 "aparece en el texto del informe anual."},
        {"pattern": r"24\.6\s*million",
         "subject": "Spin Premia", "subject_type": "Formato",
         "relation": "mide", "object": "Usuarios Activos",
         "object_type": "MetricaOperativa",
         "valor": 24.6, "unidad": "millones de usuarios", "variacion_pct": 27.5,
         "periodo": "2024",
         "nota": "El +27.5% viene del comunicado 4T24 (vs 4T23); no "
                 "aparece en el texto del informe anual."},
        {"pattern": r"1\.3\s*million", "needs": r"Juntos\+",
         "subject": "Coca-Cola FEMSA", "subject_type": "Division",
         "relation": "mide", "object": "Clientes Juntos+",
         "object_type": "MetricaOperativa",
         "valor": 1.3, "unidad": "millones de clientes", "periodo": "2024"},
        {"pattern": r"volume\s+(?:increased|grew)\s+4\.4%",
         "needs": r"Coca-Cola\s+FEMSA",
         "subject": "Coca-Cola FEMSA", "subject_type": "Division",
         "relation": "mide", "object": "Volumen",
         "object_type": "MetricaOperativa",
         "valor": 4.4, "unidad": "% crecimiento vs 2023", "periodo": "2024"},
    ]

    def _inline_build_triple(rule, page, source_label):
        t = {
            "subject": rule["subject"],
            "subject_type": rule["subject_type"],
            "relation": rule["relation"],
            "object": rule["object"],
            "object_type": rule["object_type"],
            "fuente": {"documento": source_label, "pagina": page},
        }
        for k in ("valor", "unidad", "variacion_pct", "periodo", "nota"):
            if k in rule:
                t[k] = rule[k]
        return t

    def _inline_rule_triples(chunk, source_label):
        text = chunk["text"]
        page = chunk["page"]
        out = []
        for rule in _INLINE_RULES:
            needs = rule.get("needs")
            if needs and not re.search(needs, text):
                continue
            for _m in re.finditer(rule["pattern"], text):
                out.append(_inline_build_triple(rule, page, source_label))
        # 571 estaciones solo si la página menciona OXXO GAS.
        if re.search(r"\b571\b", text) and re.search(r"OXXO\s*GAS", text, re.I):
            out.append(_inline_build_triple(
                {"subject": "Combustibles", "subject_type": "Division",
                 "relation": "mide", "object": "Estaciones",
                 "object_type": "MetricaOperativa", "valor": 571,
                 "unidad": "estaciones", "periodo": "2024"},
                page, source_label))
        seen, dedup = set(), []
        for t in out:
            key = (t["subject"], t["relation"], t["object"], t.get("valor"))
            if key not in seen:
                seen.add(key)
                dedup.append(t)
        return dedup

    _tc_all = []
    for _c in _tc_chunks:
        _tc_all.extend(_inline_rule_triples(_c, _tc_label))
    _tc_seen, _tc_triples = set(), []
    for t in _tc_all:
        key = (t["subject"], t["relation"], t["object"], t.get("valor"))
        if key not in _tc_seen:
            _tc_seen.add(key)
            _tc_triples.append(t)
    with open(_TRIPLES_OUT, "w", encoding="utf-8") as fh:
        for t in _tc_triples:
            fh.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"{len(_tc_triples)} tripletas candidatas -> {_TRIPLES_OUT}")
    print("NOTA: las tripletas son candidatas: revísalas antes de cargarlas al grafo.")
    for t in _tc_triples:
        print(f"  p.{t['fuente']['pagina']}: ({t['subject']}, {t['relation']}, "
              f"{t['object']} = {t.get('valor')} {t.get('unidad', '')})")

## Verificación: 151 páginas y la cifra clave

In [ ]:
import json

chunks = [json.loads(l) for l in open(REPO + "/pipeline/data/chunks.jsonl")]
print("paginas:", len(chunks), "| con texto:", sum(1 for c in chunks if c["chars"] > 0))
hits = [c["page"] for c in chunks if "781,585" in c["text"]]
print("781,585 aparece en paginas:", hits)
print("ejemplo p.11:", chunks[10]["text"][:300].replace("\n", " "))

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El PDF (~19 MB) y `chunks.jsonl` quedan en Drive: son los insumos que los notebooks 02–08 pedirán primero (antes de subirlos o generarlos).

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('pipeline/data/chunks.jsonl', 'chunks.jsonl (151 páginas extraídas)', True), ('pipeline/data/FEMSA_2024_Annual_Report_EN.pdf', 'PDF del informe (inglés)', False), ('pipeline/extracted_triples.jsonl', '13 tripletas candidatas por reglas (pendientes de revisión)', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    # {SUF}/{N_Q} se resuelven con los parametros del notebook (EVAL_SET);
    # en notebooks sin esos parametros valen "" y 12.
    rel = rel.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    label = label.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)